In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 28


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 2.197641499340534
Epoch 2/100, Loss: 2.1034573912620544
Epoch 3/100, Loss: 2.268039681017399
Epoch 4/100, Loss: 1.9000130593776703
Epoch 5/100, Loss: 2.185005411505699
Epoch 6/100, Loss: 1.993408903479576
Epoch 7/100, Loss: 2.0773750990629196
Epoch 8/100, Loss: 2.095249779522419
Epoch 9/100, Loss: 2.0037931725382805
Epoch 10/100, Loss: 2.0640319287776947
Epoch 11/100, Loss: 2.0715248361229897
Epoch 12/100, Loss: 2.144304037094116
Epoch 13/100, Loss: 2.0215093791484833
Epoch 14/100, Loss: 2.008163385093212
Epoch 15/100, Loss: 2.10538037866354
Epoch 16/100, Loss: 2.0341575890779495


Epoch 17/100, Loss: 2.1124786883592606
Epoch 18/100, Loss: 2.157105468213558
Epoch 19/100, Loss: 1.9844173341989517
Epoch 20/100, Loss: 2.1042960956692696
Epoch 21/100, Loss: 2.044911168515682
Epoch 22/100, Loss: 2.1488266810774803
Epoch 23/100, Loss: 2.0672109723091125
Epoch 24/100, Loss: 2.1177002415060997
Epoch 25/100, Loss: 2.143313191831112
Epoch 26/100, Loss: 2.2186058089137077
Epoch 27/100, Loss: 2.1570766866207123
Epoch 28/100, Loss: 2.055360049009323
Epoch 29/100, Loss: 2.012472502887249
Epoch 30/100, Loss: 2.1227529644966125
Epoch 31/100, Loss: 1.9785341694951057
Epoch 32/100, Loss: 2.122814364731312
Epoch 33/100, Loss: 2.209723763167858
Epoch 34/100, Loss: 1.9840946346521378


Epoch 35/100, Loss: 2.075393594801426
Epoch 36/100, Loss: 2.061026230454445
Epoch 37/100, Loss: 2.0370782762765884
Epoch 38/100, Loss: 2.116681769490242
Epoch 39/100, Loss: 2.051518552005291
Epoch 40/100, Loss: 1.979932963848114
Epoch 41/100, Loss: 2.0222917199134827
Epoch 42/100, Loss: 2.161152794957161
Epoch 43/100, Loss: 2.0052982419729233
Epoch 44/100, Loss: 2.0985744521021843
Epoch 45/100, Loss: 2.060255527496338
Epoch 46/100, Loss: 2.1805930137634277
Epoch 47/100, Loss: 2.0947221219539642
Epoch 48/100, Loss: 2.1263668835163116
Epoch 49/100, Loss: 2.0071008503437042
Epoch 50/100, Loss: 2.033387005329132
Epoch 51/100, Loss: 2.0291795432567596
Epoch 52/100, Loss: 2.1422637701034546


Epoch 53/100, Loss: 2.095602311193943
Epoch 54/100, Loss: 1.97611615806818
Epoch 55/100, Loss: 2.0717583373188972
Epoch 56/100, Loss: 2.1038626953959465
Epoch 57/100, Loss: 2.0998651906847954
Epoch 58/100, Loss: 2.0661893039941788
Epoch 59/100, Loss: 2.1636747494339943
Epoch 60/100, Loss: 2.240234814584255
Epoch 61/100, Loss: 2.0566466003656387
Epoch 62/100, Loss: 2.1769784539937973
Epoch 63/100, Loss: 2.007634863257408
Epoch 64/100, Loss: 2.060677945613861
Epoch 65/100, Loss: 2.0346013233065605
Epoch 66/100, Loss: 2.006625972688198
Epoch 67/100, Loss: 1.946480743587017
Epoch 68/100, Loss: 2.155524328351021
Epoch 69/100, Loss: 2.0912512987852097
Epoch 70/100, Loss: 2.131161689758301
Epoch 71/100, Loss: 1.9261947274208069


Epoch 72/100, Loss: 2.026010826230049
Epoch 73/100, Loss: 2.0376590341329575
Epoch 74/100, Loss: 2.0950805321335793
Epoch 75/100, Loss: 2.1233443692326546
Epoch 76/100, Loss: 2.1298495829105377
Epoch 77/100, Loss: 2.084210619330406
Epoch 78/100, Loss: 2.1411179900169373
Epoch 79/100, Loss: 2.008297950029373
Epoch 80/100, Loss: 1.9695505499839783
Epoch 81/100, Loss: 2.088477559387684
Epoch 82/100, Loss: 2.1283552274107933
Epoch 83/100, Loss: 1.9960079416632652
Epoch 84/100, Loss: 2.098350666463375
Epoch 85/100, Loss: 2.0622854456305504
Epoch 86/100, Loss: 2.103075250983238
Epoch 87/100, Loss: 2.0626955181360245
Epoch 88/100, Loss: 2.0692440271377563
Epoch 89/100, Loss: 2.175697334110737
Epoch 90/100, Loss: 2.1663927361369133


...

Epoch 14/100, Loss: 3.002586677670479
Epoch 15/100, Loss: 2.8854245841503143
Epoch 16/100, Loss: 3.281598888337612
Epoch 17/100, Loss: 3.7681609392166138
Epoch 18/100, Loss: 3.0966435074806213
Epoch 19/100, Loss: 3.1095192283391953
Epoch 20/100, Loss: 2.9145189076662064
Epoch 21/100, Loss: 4.053644925355911
Epoch 22/100, Loss: 3.2476799339056015
Epoch 23/100, Loss: 2.958944760262966
Epoch 24/100, Loss: 3.431309148669243
Epoch 25/100, Loss: 3.053533412516117
Epoch 26/100, Loss: 3.348450519144535
Epoch 27/100, Loss: 3.1383545100688934
Epoch 28/100, Loss: 3.0817159712314606
Epoch 29/100, Loss: 3.1698731034994125
Epoch 30/100, Loss: 3.314773142337799
Epoch 31/100, Loss: 3.1730280071496964


Epoch 32/100, Loss: 3.2552197873592377
Epoch 33/100, Loss: 3.1878790855407715
Epoch 34/100, Loss: 3.094450607895851
Epoch 35/100, Loss: 3.4356422126293182
Epoch 36/100, Loss: 3.2057368010282516
Epoch 37/100, Loss: 3.031627871096134
Epoch 38/100, Loss: 3.17599555850029
Epoch 39/100, Loss: 3.069014571607113
Epoch 40/100, Loss: 3.207079455256462
Epoch 41/100, Loss: 3.239220291376114
Epoch 42/100, Loss: 3.417769968509674
Epoch 43/100, Loss: 3.024542883038521
Epoch 44/100, Loss: 3.658106580376625
Epoch 45/100, Loss: 2.957817792892456
Epoch 46/100, Loss: 3.0554649010300636
Epoch 47/100, Loss: 3.070489540696144
Epoch 48/100, Loss: 2.9114611372351646
Epoch 49/100, Loss: 3.115012049674988


Epoch 50/100, Loss: 3.4526122510433197
Epoch 51/100, Loss: 3.161310702562332
Epoch 52/100, Loss: 3.1628807485103607
Epoch 53/100, Loss: 3.1698489636182785
Epoch 54/100, Loss: 3.1389021426439285
Epoch 55/100, Loss: 3.2719816416502
Epoch 56/100, Loss: 2.9701423943042755
Epoch 57/100, Loss: 3.0663625299930573
Epoch 58/100, Loss: 3.2816782370209694
Epoch 59/100, Loss: 3.0453875362873077
Epoch 60/100, Loss: 2.977781556546688
Epoch 61/100, Loss: 3.010690361261368
Epoch 62/100, Loss: 3.29070732742548
Epoch 63/100, Loss: 3.0745151937007904
Epoch 64/100, Loss: 2.9713243395090103
Epoch 65/100, Loss: 3.1959098801016808
Epoch 66/100, Loss: 3.497666671872139
Epoch 67/100, Loss: 3.099194511771202


Epoch 68/100, Loss: 2.972221925854683
Epoch 69/100, Loss: 2.9578044340014458
Epoch 70/100, Loss: 3.150180399417877
Epoch 71/100, Loss: 3.146047279238701
Epoch 72/100, Loss: 3.403188556432724
Epoch 73/100, Loss: 3.3454139977693558
Epoch 74/100, Loss: 3.081625320017338
Epoch 75/100, Loss: 3.182291731238365
Epoch 76/100, Loss: 3.3790200501680374
Epoch 77/100, Loss: 3.0498039722442627
Epoch 78/100, Loss: 3.4634112417697906
Epoch 79/100, Loss: 2.8468977957963943
Epoch 80/100, Loss: 2.9797577559947968
Epoch 81/100, Loss: 2.889573812484741
Epoch 82/100, Loss: 3.3207975029945374
Epoch 83/100, Loss: 3.3415857404470444
Epoch 84/100, Loss: 3.0566868036985397
Epoch 85/100, Loss: 3.025391273200512


Epoch 86/100, Loss: 3.4018818587064743
Epoch 87/100, Loss: 3.1833122596144676
Epoch 88/100, Loss: 3.0944306030869484
Epoch 89/100, Loss: 3.2491058707237244
Epoch 90/100, Loss: 2.8912365436553955
Epoch 91/100, Loss: 3.3020853996276855
Epoch 92/100, Loss: 2.8503749296069145
Epoch 93/100, Loss: 3.343048021197319
Epoch 94/100, Loss: 3.28144271671772
Epoch 95/100, Loss: 3.206214025616646
Epoch 96/100, Loss: 3.0208367705345154
Epoch 97/100, Loss: 2.9983511865139008
Epoch 98/100, Loss: 3.312765471637249
Epoch 99/100, Loss: 3.2777613699436188
Epoch 100/100, Loss: 3.060284301638603
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.7386
